<div style="background:#1a73e8;padding:20px;border-radius:10px;margin-bottom:10px">
<h1 style="color:white;text-align:center">⚙️ 02 — Préparation des Données</h1>
<p style="color:white;font-size:15px;text-align:center">
Nettoyage · Fenêtres glissantes · Split Train/Val/Test par batterie · Normalisation
</p></div>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler

df = pd.read_csv('data/battery_health_dataset.csv')
print(f"✅ Données chargées : {df.shape}")
df.head(3)

✅ Données chargées : (29180, 7)


,Voltage_measured,Current_measured,Temperature_measured,SoC,cycle_number,battery_id,SoH
0,3.964430,-0.912319,5.678270,97.699721,1,B0047,83.626322
1,3.843126,-0.995334,6.408629,92.510061,1,B0047,83.626322
2,3.796380,-0.995601,7.576325,87.422980,1,B0047,83.626322


## 🧹 Étape 1 — Nettoyage et vérification de l'ordre temporel

In [2]:
df_clean = df.copy()

# 1. Suppression SoH > 100% (physiquement impossible)
n_before = len(df_clean)
df_clean = df_clean[df_clean['SoH'] <= 100.0]
print(f"SoH > 100% supprimés : {n_before - len(df_clean)} lignes")

# 2. Suppression tensions hors plage physique [2.0V, 4.5V]
n_before = len(df_clean)
df_clean = df_clean[(df_clean['Voltage_measured'] >= 2.0) &
                    (df_clean['Voltage_measured'] <= 4.5)]
print(f"Tension hors [2.0V, 4.5V] supprimés : {n_before - len(df_clean)} lignes")

# 3. Suppression températures > 60°C (détectées dans 01_exploration : 160 lignes)
#    Physiquement douteuses pour une Li-ion en conditions normales
n_before = len(df_clean)
df_clean = df_clean[df_clean['Temperature_measured'] <= 60.0]
print(f"Température > 60°C supprimés       : {n_before - len(df_clean)} lignes")

# Tri temporel garanti
df_clean = df_clean.sort_values(['battery_id', 'cycle_number']).reset_index(drop=True)

print(f"\n✅ Dataset nettoyé : {df_clean.shape[0]:,} lignes")
print(f"   Batteries conservées : {df_clean['battery_id'].nunique()}")
print(f"   Total supprimé       : {len(df) - len(df_clean)} lignes ({(len(df)-len(df_clean))/len(df)*100:.1f}%)")

SoH > 100% supprimés : 140 lignes
Tension hors [2.0V, 4.5V] supprimés : 0 lignes
Température > 60°C supprimés       : 160 lignes

✅ Dataset nettoyé : 28,880 lignes
   Batteries conservées : 24
   Total supprimé       : 300 lignes (1.0%)


## 🎯 Étape 2 — Définition des variables et comparaison des tailles de fenêtre

In [3]:
# Variables d'entrée (X) : séquences temporelles multivariées — conformes à l'énoncé
FEATURES = ['Voltage_measured', 'Current_measured',
            'Temperature_measured', 'SoC', 'cycle_number']
TARGET   = 'SoH'

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

def build_windows(df_c, features, target, window):
    X_list, y_list, bat_list = [], [], []
    for (bat_id, cyc_num), group in df_c.groupby(['battery_id', 'cycle_number']):
        group = group.reset_index(drop=True)
        soh   = group[target].iloc[0]
        vals  = group[features].values
        if len(vals) < window:
            continue
        for i in range(len(vals) - window + 1):
            X_list.append(vals[i : i + window])
            y_list.append(soh)
            bat_list.append(bat_id)
    return np.array(X_list), np.array(y_list), np.array(bat_list)

def extract_stats(X_3d):
    return np.hstack([X_3d.mean(axis=1), X_3d.std(axis=1),
                      X_3d.min(axis=1),  X_3d.max(axis=1)])

# Split fixe pour la comparaison (même seed que la préparation finale)
np.random.seed(42)
all_bats_tmp = sorted(df_clean['battery_id'].unique())
shuffled_tmp = all_bats_tmp.copy()
np.random.shuffle(shuffled_tmp)
test_tmp  = shuffled_tmp[:3]
val_tmp   = shuffled_tmp[3:6]
train_tmp = shuffled_tmp[6:]

print("── Comparaison des tailles de fenêtre glissante (Ridge rapide) ──")
print(f"{'WINDOW':<10} {'Fenêtres/cycle':<18} {'Total':<14} {'MAE val (%)':<16} {'R² val':<10}")
print("─" * 70)

results_window = {}
for w in [5, 10, 15]:
    X_w, y_w, bat_w = build_windows(df_clean, FEATURES, TARGET, w)
    tr_m = np.isin(bat_w, train_tmp)
    va_m = np.isin(bat_w, val_tmp)

    sc_X = MinMaxScaler()
    n_f = len(FEATURES)
    X_tr = sc_X.fit_transform(X_w[tr_m].reshape(-1, n_f)).reshape(X_w[tr_m].shape)
    X_va = sc_X.transform(X_w[va_m].reshape(-1, n_f)).reshape(X_w[va_m].shape)

    sc_y = StandardScaler()
    y_tr_sc = sc_y.fit_transform(y_w[tr_m].reshape(-1,1)).ravel()

    ridge_w = Ridge(alpha=1.0)
    ridge_w.fit(extract_stats(X_tr), y_tr_sc)

    y_va_pred = sc_y.inverse_transform(ridge_w.predict(extract_stats(X_va)).reshape(-1,1)).ravel()
    y_va_real = y_w[va_m]

    mae_w = mean_absolute_error(y_va_real, y_va_pred)
    r2_w  = r2_score(y_va_real, y_va_pred)
    n_fen = 20 - w + 1
    results_window[w] = {'mae': mae_w, 'r2': r2_w}
    best_mark = " ← ★" if w == 10 else ""
    print(f"  WINDOW={w:<5} {n_fen:<18} {X_w.shape[0]:<14,} {mae_w:<16.4f} {r2_w:<10.4f}{best_mark}")

print()
print("✅ WINDOW=10 sélectionnée (meilleur compromis MAE / augmentation données)")
WINDOW = 10
print(f"   Features : {FEATURES}")
print(f"   Cible    : {TARGET}")
print(f"   Fenêtre  : {WINDOW} bins (50% du cycle de 20 bins)")
print(f"   Fenêtres/cycle : {20 - WINDOW + 1}")

── Comparaison des tailles de fenêtre glissante (Ridge rapide) ──
WINDOW     Fenêtres/cycle     Total          MAE val (%)      R² val    
──────────────────────────────────────────────────────────────────────
  WINDOW=5     16                 23,072         3.9186           -0.0096   
  WINDOW=10    11                 15,812         4.2349           -0.1881    ← ★
  WINDOW=15    6                  8,558          4.2693           -0.1832   

✅ WINDOW=10 sélectionnée (meilleur compromis MAE / augmentation données)
   Features : ['Voltage_measured', 'Current_measured', 'Temperature_measured', 'SoC', 'cycle_number']
   Cible    : SoH
   Fenêtre  : 10 bins (50% du cycle de 20 bins)
   Fenêtres/cycle : 11


## 🔄 Étape 3 — Construction des fenêtres glissantes

In [4]:
X_list, y_list, meta = [], [], []

for (bat_id, cyc_num), group in df_clean.groupby(['battery_id', 'cycle_number']):
    group = group.reset_index(drop=True)
    soh   = group[TARGET].iloc[0]
    vals  = group[FEATURES].values

    if len(vals) < WINDOW:
        continue

    for i in range(len(vals) - WINDOW + 1):
        X_list.append(vals[i : i + WINDOW])
        y_list.append(soh)
        meta.append({'battery_id': bat_id, 'cycle_number': cyc_num})

X_all    = np.array(X_list)
y_all    = np.array(y_list)
meta_all = pd.DataFrame(meta)

print(f"✅ Fenêtres créées")
print(f"   Forme X : {X_all.shape}  → (échantillons, fenêtre, features)")
print(f"   Forme y : {y_all.shape}")
print(f"   SoH min/max : {y_all.min():.2f}% → {y_all.max():.2f}%")

✅ Fenêtres créées
   Forme X : (15812, 10, 5)  → (échantillons, fenêtre, features)
   Forme y : (15812,)
   SoH min/max : 70.02% → 99.73%


## ✂️ Étape 4 — Split Train / Validation / Test par batterie

**Principe :** On divise par **batterie entière** pour éviter tout data leakage.
- **Train (18 batteries)** : apprentissage du modèle
- **Validation (3 batteries)** : surveillance pendant l'entraînement (EarlyStopping)
- **Test (3 batteries)** : évaluation finale — ces batteries ne seront JAMAIS vues

**Pourquoi pas `validation_split=0.1` dans model.fit() ?**  
Cette option découpe au hasard parmi toutes les fenêtres → des fenêtres de la MÊME batterie  
se retrouvent en train ET validation → val_loss artificiellement bonne (data leakage).

In [5]:
np.random.seed(42)
all_bats  = sorted(df_clean['battery_id'].unique())
n_bats    = len(all_bats)

n_test    = max(2, round(n_bats * 0.125))
n_val     = max(2, round(n_bats * 0.125))

shuffled  = all_bats.copy()
np.random.shuffle(shuffled)

test_bats  = shuffled[:n_test]
val_bats   = shuffled[n_test : n_test + n_val]
train_bats = shuffled[n_test + n_val:]

print(f"✅ Split par batterie")
print(f"   Train ({len(train_bats)} batteries) : {sorted(train_bats)}")
print(f"   Val   ({len(val_bats)} batteries)  : {sorted(val_bats)}")
print(f"   Test  ({len(test_bats)} batteries) : {sorted(test_bats)}")

train_mask = meta_all['battery_id'].isin(train_bats).values
val_mask   = meta_all['battery_id'].isin(val_bats).values
test_mask  = meta_all['battery_id'].isin(test_bats).values

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val,   y_val   = X_all[val_mask],   y_all[val_mask]
X_test,  y_test  = X_all[test_mask],  y_all[test_mask]

print(f"\n   Train : {X_train.shape[0]:,} fenêtres | plage SoH: {y_train.min():.1f}% – {y_train.max():.1f}%")
print(f"   Val   : {X_val.shape[0]:,} fenêtres   | plage SoH: {y_val.min():.1f}% – {y_val.max():.1f}%")
print(f"   Test  : {X_test.shape[0]:,} fenêtres  | plage SoH: {y_test.min():.1f}% – {y_test.max():.1f}%")

✅ Split par batterie
   Train (18 batteries) : ['B0006', 'B0007', 'B0018', 'B0025', 'B0026', 'B0027', 'B0028', 'B0031', 'B0033', 'B0034', 'B0036', 'B0038', 'B0040', 'B0043', 'B0044', 'B0046', 'B0047', 'B0048']
   Val   (3 batteries)  : ['B0030', 'B0032', 'B0042']
   Test  (3 batteries) : ['B0005', 'B0029', 'B0039']

   Train : 12,252 fenêtres | plage SoH: 70.0% – 99.7%
   Val   : 1,404 fenêtres   | plage SoH: 70.6% – 94.4%
   Test  : 2,156 fenêtres  | plage SoH: 70.2% – 92.5%


## 📐 Étape 5 — Normalisation

**Features X** : `MinMaxScaler` → plage [0, 1]  
**Cible y** : `StandardScaler` → moyenne 0, écart-type 1  
*(plus robuste que MinMaxScaler pour y car il n'est pas sensible aux valeurs hors plage train)*  

**Règle anti-leakage** : fit UNIQUEMENT sur train, transform sur val et test.

In [6]:
n_feat = len(FEATURES)

# Normalisation X : MinMaxScaler (features bornées)
scaler_X = MinMaxScaler()
X_train_sc = scaler_X.fit_transform(X_train.reshape(-1, n_feat)).reshape(X_train.shape)
X_val_sc   = scaler_X.transform(X_val.reshape(-1, n_feat)).reshape(X_val.shape)
X_test_sc  = scaler_X.transform(X_test.reshape(-1, n_feat)).reshape(X_test.shape)

# Normalisation y : StandardScaler (plus robuste pour une valeur continue non bornée)
scaler_y = StandardScaler()
y_train_sc = scaler_y.fit_transform(y_train.reshape(-1,1)).ravel()
y_val_sc   = scaler_y.transform(y_val.reshape(-1,1)).ravel()
y_test_sc  = scaler_y.transform(y_test.reshape(-1,1)).ravel()

print("✅ Normalisation appliquée (fit sur train uniquement)")
print(f"   X_train normalisé : [{X_train_sc.min():.3f}, {X_train_sc.max():.3f}]")
print(f"   y_train — μ={y_train_sc.mean():.3f}, σ={y_train_sc.std():.3f} (StandardScaler)")
print(f"   SoH train original : {y_train.mean():.2f}% ± {y_train.std():.2f}%")

✅ Normalisation appliquée (fit sur train uniquement)
   X_train normalisé : [0.000, 1.000]
   y_train — μ=-0.000, σ=1.000 (StandardScaler)
   SoH train original : 81.83% ± 7.12%


## 💾 Étape 6 — Sauvegarde du pipeline

In [7]:
import pickle

data_prepared = {
    'X_train_sc': X_train_sc, 'X_val_sc': X_val_sc,   'X_test_sc': X_test_sc,
    'y_train_sc': y_train_sc, 'y_val_sc': y_val_sc,   'y_test_sc': y_test_sc,
    'y_train':    y_train,    'y_val':    y_val,       'y_test':    y_test,
    'scaler_X':   scaler_X,   'scaler_y': scaler_y,
    'meta_all':   meta_all,
    'train_bats': train_bats, 'val_bats': val_bats,   'test_bats': test_bats,
    'train_mask': train_mask, 'val_mask': val_mask,   'test_mask': test_mask,
    'FEATURES':   FEATURES,   'TARGET':   TARGET,     'WINDOW':    WINDOW
}

with open('data/prepared_data.pkl', 'wb') as f:
    pickle.dump(data_prepared, f)

print("✅ Données sauvegardées dans data/prepared_data.pkl")
print("   → Prêt pour 03_modele_lstm.ipynb 🚀")

✅ Données sauvegardées dans data/prepared_data.pkl
   → Prêt pour 03_modele_lstm.ipynb 🚀
